In [2]:
import torch
from torch import nn
import torchvision
from torchvision import datasets,transforms
import numpy as np
import matplotlib.pyplot as plt
import torch.utils.data as Data
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [3]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),               #32×32
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))  
])
train_dataset=datasets.FashionMNIST(root="神经网络实践/data",train=True,transform=transform,download=True)
test_dataset=datasets.FashionMNIST(root="神经网络实践/data",train=False,transform=transform,download=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2)


In [4]:
class LeNet(nn.Module):
    def __init__(self,num_classes=10):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,6,kernel_size=5,padding=0),
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(6,16,kernel_size=5,padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
        )
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*5*5,120),
            nn.BatchNorm1d(120),
            nn.ReLU(),
            nn.Linear(120,84),
            nn.BatchNorm1d(84),
            nn.ReLU(),
            nn.Linear(84,num_classes),
        )
    def forward(self,x):
        x=self.features(x)
        x=self.classifier(x)
        return x
model=LeNet()
        

In [5]:
loss=nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            loss_sum += loss(out, y).item() * y.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            total    += y.size(0)
    return loss_sum / total, correct / total
EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch_idx, (x, y) in enumerate(train_loader):
        out = model(x)
        l = loss(out, y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        if (batch_idx + 1) % 200 == 0:
            print(f'epoch {epoch}  step {batch_idx+1}/{len(train_loader)}  loss={l.item():.4f}')
    tr_loss, tr_acc = evaluate(train_loader)
    te_loss, te_acc = evaluate(test_loader)
    print(f'>> epoch {epoch}  train_loss={tr_loss:.4f}  test_acc={te_acc:.4f}\n')

epoch 1  step 200/938  loss=0.5053
epoch 1  step 400/938  loss=0.4441
epoch 1  step 600/938  loss=0.4112
epoch 1  step 800/938  loss=0.2794
>> epoch 1  train_loss=0.2846  test_acc=0.8825

epoch 2  step 200/938  loss=0.3766
epoch 2  step 400/938  loss=0.2153
epoch 2  step 600/938  loss=0.1937
epoch 2  step 800/938  loss=0.1919
>> epoch 2  train_loss=0.2588  test_acc=0.8896

epoch 3  step 200/938  loss=0.1150
epoch 3  step 400/938  loss=0.2567
epoch 3  step 600/938  loss=0.3493
epoch 3  step 800/938  loss=0.1585
>> epoch 3  train_loss=0.2197  test_acc=0.8998

epoch 4  step 200/938  loss=0.1964
epoch 4  step 400/938  loss=0.1749
epoch 4  step 600/938  loss=0.1349
epoch 4  step 800/938  loss=0.2670
>> epoch 4  train_loss=0.2174  test_acc=0.8926

epoch 5  step 200/938  loss=0.1537
epoch 5  step 400/938  loss=0.0678
epoch 5  step 600/938  loss=0.2167
epoch 5  step 800/938  loss=0.2170
>> epoch 5  train_loss=0.2149  test_acc=0.8932

epoch 6  step 200/938  loss=0.1418
epoch 6  step 400/938  lo